# Cybersecurity Network Behaviour Analytics
## Project 1 — KNN-based Network Anomaly Detection

**Challenge:** Cybersecurity — What Does Normal Network Behaviour Look Like?

### Important dataset note
The supplied challenge specification defines the required fields and tasks but does not include actual network-flow rows. Therefore, this project generates a **synthetic anonymized network-flow dataset** for demonstration and development.

The synthetic validation flag is used only to benchmark the implementation. It is **not presented as a real attack label**.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)

BASE = Path("..")
DATA = BASE / "data"
VIZ = BASE / "visualizations"
RESULTS = BASE / "results"

for p in [DATA, VIZ, RESULTS]:
    p.mkdir(exist_ok=True)

print("Project folders ready.")


## 1. Generate the synthetic network-flow dataset

In [ ]:
n = 1500

protocols = ["TCP", "UDP", "HTTP", "HTTPS", "DNS", "SSH"]
sources = [f"DEV-{i:03d}" for i in range(1, 51)]
destinations = [f"SRV-{i:03d}" for i in range(1, 31)]

protocol = np.random.choice(protocols, n, p=[0.30, 0.20, 0.15, 0.15, 0.10, 0.10])
source = np.random.choice(sources, n)
destination = np.random.choice(destinations)
timestamp = pd.date_range("2026-01-01 00:00:00", periods=n, freq="20min")

bytes_transferred = np.random.lognormal(mean=8.0, sigma=0.75, size=n)
duration = np.random.lognormal(mean=2.0, sigma=0.65, size=n)
connection_frequency = np.random.poisson(lam=8, size=n).astype(float) + 1

# Hidden synthetic validation truth; not part of the challenge-style input.
is_unusual = np.random.rand(n) < 0.08

bytes_transferred[is_unusual] *= np.random.uniform(5, 15, is_unusual.sum())
duration[is_unusual] *= np.random.uniform(0.15, 7, is_unusual.sum())
connection_frequency[is_unusual] *= np.random.uniform(4, 10, is_unusual.sum())

df = pd.DataFrame({
    "Flow_ID": [f"F{i:05d}" for i in range(1, n + 1)],
    "Timestamp": timestamp,
    "Source": source,
    "Destination": destination,
    "Bytes_Transferred": np.round(bytes_transferred, 2),
    "Duration": np.round(duration, 2),
    "Protocol": protocol,
    "Connection_Frequency": np.round(connection_frequency, 2)
})

validation_truth = pd.DataFrame({
    "Flow_ID": df["Flow_ID"],
    "Validation_Unusual": is_unusual.astype(int)
})

df.to_csv(DATA / "network_flow.csv", index=False)
validation_truth.to_csv(DATA / "synthetic_validation_truth.csv", index=False)

print(df.head())
print("\nShape:", df.shape)


## 2. Data cleaning and validation

In [ ]:
# Add controlled quality issues to demonstrate the cleaning workflow.
raw = df.copy()
raw = pd.concat([raw, raw.iloc[[10, 25, 100]]], ignore_index=True)
raw.loc[5, "Bytes_Transferred"] = np.nan
raw.loc[17, "Duration"] = np.nan
raw.loc[33, "Protocol"] = None

print("Missing values before cleaning:")
print(raw.isnull().sum())
print("\nDuplicate rows:", raw.duplicated().sum())

raw.to_csv(DATA / "network_flow_raw.csv", index=False)

clean = raw.drop_duplicates().copy()
clean["Timestamp"] = pd.to_datetime(clean["Timestamp"], errors="coerce")

numeric_cols = ["Bytes_Transferred", "Duration", "Connection_Frequency"]
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors="coerce")
    clean[col] = clean[col].fillna(clean[col].median())

clean["Protocol"] = clean["Protocol"].fillna(clean["Protocol"].mode()[0])
clean = clean.dropna(subset=["Timestamp", "Source", "Destination"]).reset_index(drop=True)

clean.to_csv(DATA / "network_flow_clean.csv", index=False)

print("\nAfter cleaning:")
print(clean.info())
print("\nMissing values:")
print(clean.isnull().sum())


## 3. Establish normal network behaviour

In [ ]:
features = ["Bytes_Transferred", "Duration", "Connection_Frequency"]

print("Descriptive statistics:")
display(clean[features].describe())

print("\nProtocol distribution:")
display(clean["Protocol"].value_counts())

print("\nNormal baseline — median values:")
display(clean[features].median())


## 4. Protocol and time analysis

In [ ]:
protocol_summary = clean.groupby("Protocol").agg(
    Flows=("Flow_ID", "count"),
    Avg_Bytes=("Bytes_Transferred", "mean"),
    Avg_Duration=("Duration", "mean"),
    Avg_Connection_Frequency=("Connection_Frequency", "mean")
).sort_values("Flows", ascending=False)

clean["Hour"] = clean["Timestamp"].dt.hour

hour_summary = clean.groupby("Hour").agg(
    Flows=("Flow_ID", "count"),
    Avg_Bytes=("Bytes_Transferred", "mean"),
    Avg_Connection_Frequency=("Connection_Frequency", "mean")
)

display(protocol_summary)
display(hour_summary.head())


## 5. Highly active source devices

In [ ]:
device_activity = clean.groupby("Source").agg(
    Flow_Count=("Flow_ID", "count"),
    Total_Bytes=("Bytes_Transferred", "sum"),
    Avg_Connection_Frequency=("Connection_Frequency", "mean")
).sort_values("Flow_Count", ascending=False)

display(device_activity.head(10))


## 6. Four required visualisations

In [ ]:
# 1. Protocol distribution
plt.figure(figsize=(10,5))
clean["Protocol"].value_counts().plot(kind="bar")
plt.title("Network Flows by Protocol")
plt.xlabel("Protocol")
plt.ylabel("Number of Flows")
plt.tight_layout()
plt.savefig(VIZ / "01_protocol_distribution.png", dpi=180)
plt.show()

# 2. Bytes vs connection frequency
plt.figure(figsize=(10,5))
plt.scatter(clean["Bytes_Transferred"], clean["Connection_Frequency"], alpha=0.55)
plt.title("Bytes Transferred vs Connection Frequency")
plt.xlabel("Bytes Transferred")
plt.ylabel("Connection Frequency")
plt.tight_layout()
plt.savefig(VIZ / "02_bytes_vs_frequency.png", dpi=180)
plt.show()

# 3. Hourly activity
plt.figure(figsize=(10,5))
hour_summary["Flows"].plot(kind="line", marker="o")
plt.title("Network Activity by Hour")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Flows")
plt.xticks(range(0,24))
plt.tight_layout()
plt.savefig(VIZ / "03_hourly_activity.png", dpi=180)
plt.show()

# 4. Top active devices
plt.figure(figsize=(10,5))
device_activity.head(10)["Flow_Count"].sort_values().plot(kind="barh")
plt.title("Top 10 Most Active Source Devices")
plt.xlabel("Number of Flows")
plt.ylabel("Source Device")
plt.tight_layout()
plt.savefig(VIZ / "04_top_active_devices.png", dpi=180)
plt.show()


## 7. KNN anomaly detection

In [ ]:
X = clean[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10
knn = NearestNeighbors(n_neighbors=k + 1)
knn.fit(X_scaled)

distances, indices = knn.kneighbors(X_scaled)

# Exclude the record itself from its neighbour-distance calculation.
clean["KNN_Anomaly_Score"] = distances[:, 1:].mean(axis=1)

# Flag the top 8% as unusual.
threshold = clean["KNN_Anomaly_Score"].quantile(0.92)
clean["KNN_Unusual"] = (clean["KNN_Anomaly_Score"] >= threshold).astype(int)

results = clean.sort_values("KNN_Anomaly_Score", ascending=False)
results.to_csv(RESULTS / "knn_anomaly_results.csv", index=False)

print("KNN:", k)
print("Anomaly threshold:", round(threshold, 4))
print("Unusual flows:", int(clean["KNN_Unusual"].sum()))
display(results.head(20)[[
    "Flow_ID","Timestamp","Source","Destination",
    "Bytes_Transferred","Duration","Protocol",
    "Connection_Frequency","KNN_Anomaly_Score","KNN_Unusual"
]])


## 8. Synthetic validation of the implementation

In [ ]:
# This benchmark uses the synthetic validation truth created during dataset generation.
# It must NOT be described as real attack labels from the challenge.

eval_df = clean.merge(validation_truth, on="Flow_ID", how="inner")

y_true = eval_df["Validation_Unusual"]
y_pred = eval_df["KNN_Unusual"]

precision = precision_score(y_true, y_pred, zero_division=0)
recall = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
cm = confusion_matrix(y_true, y_pred)

print(f"Precision: {precision:.3f}")
print(f"Recall:    {recall:.3f}")
print(f"F1-score:  {f1:.3f}")
print("\nConfusion matrix:")
print(cm)


## 9. Key findings and action plan

In [ ]:
protocol_anomaly = clean.groupby("Protocol")["KNN_Unusual"].mean().sort_values(ascending=False)
hour_anomaly = clean.groupby("Hour")["KNN_Unusual"].mean().sort_values(ascending=False)

print("KEY INSIGHTS")
print(f"1. {int(clean['KNN_Unusual'].sum())} of {len(clean)} cleaned flows were flagged as unusual.")
print(f"2. {protocol_anomaly.index[0]} had the highest unusual-flow rate in this synthetic dataset.")
print(f"3. {device_activity.index[0]} generated the highest number of flows.")
print(f"4. {device_activity['Total_Bytes'].idxmax()} generated the highest total transferred volume.")
print(f"5. Hour {int(hour_anomaly.index[0]):02d}:00 had the highest unusual-flow rate.")
print(f"6. The synthetic validation F1-score was {f1:.3f}.")
print("7. An anomaly is a deviation from the established baseline and should not automatically be labelled malicious.")

print("\nPRACTICAL ACTION PLAN")
actions = [
    "Prioritise high KNN-score flows for investigation.",
    "Review devices with repeated anomalous behaviour.",
    "Inspect unusually large transfers and high connection frequencies.",
    "Compare anomalies against normal protocol and hourly baselines.",
    "Investigate repeated source-destination pairs showing abnormal behaviour.",
    "Update the baseline periodically as network usage changes.",
    "Require contextual investigation before escalating an anomaly as a security incident."
]
for i, action in enumerate(actions, 1):
    print(f"{i}. {action}")


## Conclusion

The project establishes a baseline for network behaviour using traffic volume, duration, connection frequency, protocol and time characteristics. KNN nearest-neighbour distances are used as an anomaly score to identify records that differ strongly from nearby behavioural patterns.

**Interpretation:** an unusual record is a candidate for investigation, not proof of malicious activity.
